# 🌊 Bahari — Zanzibar Aqua Tourism RAG Pipeline

This notebook builds a full **Retrieval-Augmented Generation (RAG)** pipeline for a Zanzibar aqua-tourism assistant, step by step, matching the standard RAG diagram:

`DOCUMENTS → CHUNKS → EMBEDDINGS → VECTOR DATABASE → QUESTION → RETRIEVAL → CONTEXT → LLM → ANSWER`

**LLM provider:** [Groq](https://console.groq.com/) (fast open-weight model inference, e.g. `llama-3.3-70b-versatile`).
**Frontend:** a companion `streamlit_app.py` (in the same folder) reuses the vector store this notebook builds.

---
### Setup
1. `pip install -r requirements.txt`
2. Get a free API key at https://console.groq.com/keys
3. Copy `.env.example` to `.env` and paste your key in:
   ```
   GROQ_API_KEY=your-key-here
   ```
   Both this notebook and `streamlit_app.py` load `.env` automatically via `python-dotenv`.
   `.env` is listed in `.gitignore` — never commit it or hardcode the key in the notebook itself.

In [9]:
# Optional: run once to install dependencies (uncomment if needed)
#%pip install -q sentence-transformers faiss-cpu groq scikit-learn numpy pypdf beautifulsoup4

In [10]:
import os, glob, json, pickle, textwrap
import numpy as np
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from a local .env file if present

DOCS_DIR = "documents"          # folder of .txt knowledge files
STORE_PATH = "vector_store.pkl"  # persisted index, shared with streamlit_app.py
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
GROQ_MODEL = "llama-3.3-70b-versatile"


## 1. Documents
**What happens:** load raw knowledge from files (here: plain `.txt` files — swap in `pypdf`/`pdfplumber` for PDFs, or `BeautifulSoup` for scraped web pages).

Each file in `documents/` covers one Zanzibar aqua-tourism topic (dive sites, tours, seasons, prices, safety, etc.).

In [11]:
def load_documents(docs_dir=DOCS_DIR):
    """Load .txt documents. First line = title, rest = body text."""
    documents = []
    for path in sorted(glob.glob(os.path.join(docs_dir, "*.txt"))):
        with open(path, "r", encoding="utf-8") as f:
            content = f.read().strip()
        title, _, body = content.partition("\n\n")
        documents.append({"title": title.strip(), "text": body.strip(), "source": os.path.basename(path)})
    return documents

# --- Alternative loaders for other formats ---
# from pypdf import PdfReader
# def load_pdf(path):
#     reader = PdfReader(path)
#     return "\n".join(page.extract_text() or "" for page in reader.pages)
#
# from bs4 import BeautifulSoup
# def load_html(path):
#     soup = BeautifulSoup(open(path), "html.parser")
#     return soup.get_text(separator="\n")

documents = load_documents()
print(f"Loaded {len(documents)} documents")
documents[0]

Loaded 24 documents


{'title': 'Mnemba Atoll',
 'text': "Mnemba Atoll off the northeast coast, near Nungwi and Matemwe, is Zanzibar's best-known snorkeling and diving spot. It's a protected marine conservation area with clear water, coral reef walls, sea turtles, and reef fish. Most visitors reach it by boat on a half-day trip from nearby beach towns. A conservation fee is charged separately from the boat trip cost.",
 'source': '01_mnemba_atoll.txt'}

## 2. Chunking
**What happens:** break documents into smaller pieces so retrieval can return focused, relevant passages instead of whole documents.

These source documents are already short (one topic each), so we use light word-based chunking with overlap — the same approach `langchain`'s `RecursiveCharacterTextSplitter` uses under the hood. For longer documents (e.g. PDFs), lower `chunk_size` and keep some `overlap`.

In [12]:
def chunk_text(text, chunk_size=120, overlap=20):
    """Simple word-based sliding-window chunker."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    chunks, start = [], 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

# --- Equivalent with LangChain (if installed) ---
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
# chunks = splitter.split_text(text)

chunks = []
for doc in documents:
    for i, piece in enumerate(chunk_text(doc["text"])):
        chunks.append({"title": doc["title"], "text": piece, "source": doc["source"], "chunk_id": i})

print(f"{len(documents)} documents -> {len(chunks)} chunks")
chunks[0]

24 documents -> 24 chunks


{'title': 'Mnemba Atoll',
 'text': "Mnemba Atoll off the northeast coast, near Nungwi and Matemwe, is Zanzibar's best-known snorkeling and diving spot. It's a protected marine conservation area with clear water, coral reef walls, sea turtles, and reef fish. Most visitors reach it by boat on a half-day trip from nearby beach towns. A conservation fee is charged separately from the boat trip cost.",
 'source': '01_mnemba_atoll.txt',
 'chunk_id': 0}

## 3. Embeddings
**What happens:** convert each chunk's text into a numerical vector that captures its meaning, so semantically similar text ends up close together in vector space.

Primary path: `sentence-transformers` (`all-MiniLM-L6-v2` — small, fast, good quality). If it isn't installed, this falls back to a TF-IDF vectorizer (`scikit-learn`) so the notebook still runs end-to-end — swap it out for the real embedding model in production.

In [13]:
USE_SENTENCE_TRANSFORMERS = True

try:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer(EMBED_MODEL_NAME)

    def embed_texts(texts):
        return np.array(embedder.encode(texts, normalize_embeddings=True))

    print(f"Using sentence-transformers ({EMBED_MODEL_NAME})")
except ImportError:
    USE_SENTENCE_TRANSFORMERS = False
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import normalize

    _tfidf = TfidfVectorizer(stop_words="english")
    _tfidf.fit([c["text"] for c in chunks])

    def embed_texts(texts):
        vecs = _tfidf.transform(texts).toarray()
        return normalize(vecs)

    print("sentence-transformers not installed — falling back to TF-IDF (install sentence-transformers for real semantic embeddings)")

# --- Alternative embedding providers ---
# OpenAI:  client.embeddings.create(model="text-embedding-3-small", input=texts)
# Cohere:  co.embed(texts=texts, model="embed-english-v3.0")
# Google:  genai.embed_content(model="models/text-embedding-004", content=texts)

sentence-transformers not installed — falling back to TF-IDF (install sentence-transformers for real semantic embeddings)


In [14]:
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_texts(chunk_texts)
print("Embedding matrix shape:", chunk_embeddings.shape)

Embedding matrix shape: (24, 420)


## 4. Vector Database
**What happens:** store the chunk vectors in an index that supports fast similarity search.

Primary path: `faiss` (Facebook AI Similarity Search) with a flat inner-product index — since embeddings are normalized, inner product = cosine similarity. If `faiss` isn't installed, falls back to plain NumPy cosine similarity (fine at this small scale, not meant for millions of vectors).

In [15]:
USE_FAISS = True
try:
    import faiss
    dim = chunk_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(chunk_embeddings.astype("float32"))
    print(f"Built FAISS IndexFlatIP with {index.ntotal} vectors (dim={dim})")
except ImportError:
    USE_FAISS = False
    index = None
    print("faiss not installed — falling back to NumPy cosine similarity search")

# --- Other common vector DB options ---
# Chroma:   client.get_or_create_collection('zanzibar').add(...)
# Pinecone: index.upsert(vectors=[...])
# Qdrant:   client.upsert(collection_name='zanzibar', points=[...])
# Weaviate: client.data_object.create(...)

faiss not installed — falling back to NumPy cosine similarity search


## 5. User Question
**What happens:** receive the traveler's question. In production this comes from the Streamlit UI (`streamlit_app.py`); here we use a sample question to test the pipeline.

In [16]:
user_question = "Where can I go snorkeling and how much does it usually cost?"
print(user_question)

Where can I go snorkeling and how much does it usually cost?


## 6. Question Embedding
**What happens:** convert the user's question into a vector using the *same* embedding function used for the chunks, so it lands in the same vector space.

In [17]:
question_embedding = embed_texts([user_question])
print("Question embedding shape:", question_embedding.shape)

Question embedding shape: (1, 420)


## 7. Retrieval
**What happens:** search the vector database for the chunks most similar to the question embedding.

In [18]:
def retrieve(question, k=4):
    q_vec = embed_texts([question]).astype("float32")
    if USE_FAISS:
        scores, idxs = index.search(q_vec, k)
        results = [(chunks[i], float(s)) for i, s in zip(idxs[0], scores[0]) if i != -1]
    else:
        sims = (chunk_embeddings @ q_vec[0])
        top_idx = np.argsort(-sims)[:k]
        results = [(chunks[i], float(sims[i])) for i in top_idx]
    return results

retrieved = retrieve(user_question, k=4)
for chunk, score in retrieved:
    print(f"[{score:.3f}] {chunk['title']}")

[0.239] Currents and independent snorkeling
[0.166] Mnemba Atoll
[0.137] Marine park entrance fees
[0.137] Chumbe Island Coral Park


## 8. Relevant Information (build context)
**What happens:** combine the retrieved chunks into a single context block to hand to the LLM.

In [19]:
def build_context(retrieved_chunks):
    return "\n\n".join(f"[{c['title']}]\n{c['text']}" for c, _ in retrieved_chunks)

context = build_context(retrieved)
print(context[:600] + "...")

[Currents and independent snorkeling]
Currents can be strong in channels such as around Mnemba, so snorkeling there is usually done with a guide rather than independently. Anyone snorkeling off the beach independently, particularly on the east coast, should check local tide and current conditions first.

[Mnemba Atoll]
Mnemba Atoll off the northeast coast, near Nungwi and Matemwe, is Zanzibar's best-known snorkeling and diving spot. It's a protected marine conservation area with clear water, coral reef walls, sea turtles, and reef fish. Most visitors reach it by boat on a half-day trip from ne...


## 9. LLM — Generate the Answer (Groq)
**What happens:** send the question + retrieved context to an LLM so it can understand the context and write the answer, grounded only in what was retrieved.

Uses the [Groq API](https://console.groq.com/) (`pip install groq`). Reads the key from the `GROQ_API_KEY` environment variable — set it before running this cell, don't paste it directly into the notebook.

In [20]:
SYSTEM_PROMPT = (
    "You are Bahari, a friendly, practical assistant for people planning aqua tourism activities "
    "(diving, snorkeling, boat trips, island trips) in Zanzibar. Answer using ONLY the context passages "
    "provided — do not invent details that aren't in the context. If the context doesn't cover something, "
    "say so plainly. Keep answers concise (roughly 3-6 sentences), warm, and specific to Zanzibar. Any prices "
    "mentioned are approximate and should be confirmed locally."
)

def ask_groq(question, context, model=GROQ_MODEL):
    from groq import Groq
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("GROQ_API_KEY environment variable is not set. Get a key at https://console.groq.com/keys")
    client = Groq(api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": f"{SYSTEM_PROMPT}\n\nCONTEXT:\n{context}"},
            {"role": "user", "content": question},
        ],
        temperature=0.4,
        max_tokens=500,
    )
    return response.choices[0].message.content

In [21]:
try:
    answer = ask_groq(user_question, context)
except Exception as e:
    answer = f"[Could not call Groq: {e}]"
print(answer)

[Could not call Groq: No module named 'groq']


## 10. Answer
**What happens:** display the generated response, along with the sources it was grounded in — useful for transparency and debugging retrieval quality.

In [22]:
print("Q:", user_question)
print("\nA:", textwrap.fill(answer, 100))
print("\nSources used:")
for chunk, score in retrieved:
    print(f"  - {chunk['title']}  (similarity {score:.3f})")

Q: Where can I go snorkeling and how much does it usually cost?

A: [Could not call Groq: No module named 'groq']

Sources used:
  - Currents and independent snorkeling  (similarity 0.239)
  - Mnemba Atoll  (similarity 0.166)
  - Marine park entrance fees  (similarity 0.137)
  - Chumbe Island Coral Park  (similarity 0.137)


## Persist the vector store
Save everything the Streamlit dashboard needs, so it doesn't have to re-embed documents on every launch. Re-run this cell (and the pipeline above) any time you add or edit files in `documents/`.

In [23]:
with open(STORE_PATH, "wb") as f:
    pickle.dump({
        "chunks": chunks,
        "embeddings": chunk_embeddings,
        "use_sentence_transformers": USE_SENTENCE_TRANSFORMERS,
        "embed_model_name": EMBED_MODEL_NAME,
    }, f)

print(f"Saved vector store with {len(chunks)} chunks to {STORE_PATH}")
print("Now run:  streamlit run streamlit_app.py")

Saved vector store with 24 chunks to vector_store.pkl
Now run:  streamlit run streamlit_app.py


---
### Try more questions
Re-run cells 5-10 (or just call the helper below) with different questions to explore the pipeline.

In [24]:
def rag_answer(question, k=4):
    retrieved = retrieve(question, k=k)
    context = build_context(retrieved)
    try:
        answer = ask_groq(question, context)
    except Exception as e:
        answer = f"[Could not call Groq: {e}]"
    return answer, retrieved

answer, sources = rag_answer("When's the best time of year to visit for calm seas?")
print(answer)

[Could not call Groq: No module named 'groq']
